# Problem 2. 영화 리뷰 감성 분류하기

## 문제 개요
영화 정보 서비스는 사용자가 남긴 짧은 한 줄 리뷰가 **긍정(1)인지 부정(0)인지** 자동으로 분류하려 합니다.
리뷰 텍스트로 감성을 예측하는 모델을 만드세요.

## 파일 구조
```
problem02/
├── work.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 20,000개 (label 포함)
- `data/test.csv` : 10,000개 (label 없음)

| 컬럼 | 설명 |
|---|---|
| id | 고유 번호 (`R00000` 형식) |
| review | 리뷰 본문 (최대 140자) |
| **label** | **1 = 긍정, 0 = 부정 (예측 대상)** |

### 데이터 특징
- 긍정/부정 비율은 거의 1:1입니다.
- 맞춤법 오류, 띄어쓰기 생략, 초성(ㅋㅋ, ㅠㅠ), 반복 문자가 매우 많습니다.
- 출처: NSMC(Naver Sentiment Movie Corpus, CC0) 일부를 대회용으로 재구성

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`problem02/`)에 **`submission.csv`** 로 저장

## 평가
**Accuracy (정확도)** — 높을수록 좋습니다.

## 제출 형식
`submission.csv`

| id | label |
|---|---|
| R20000 | 1 |
| R20001 | 0 |

- 컬럼명 `id`, `label`, label은 0 또는 1 정수
- 포맷이 틀리면 0점 처리됩니다.

## 제출 검증 규칙
- 테스트의 모든 ID를 정확히 한 번씩 제출합니다. 누락·중복·추가 ID는 오류이며, 행 순서는 자유입니다(채점 시 ID로 정렬합니다).
- CSV는 UTF-8(또는 UTF-8 BOM), 쉼표 구분, 헤더 포함 형식입니다. 아래 두 컬럼만 제출하며 컬럼 순서는 자유입니다.
- 비어 있는 셀, NaN, 무한대, 깨진 CSV는 오류입니다. 올바르지 않은 제출은 해당 문제 0점과 ERROR로 기록합니다.
- 최종 파일은 이 문제 폴더의 `submission.csv`입니다. 다른 이름이나 하위 폴더 파일은 수집하지 않습니다.
- 컬럼: `id,label`; label은 0 또는 1 정수입니다. 확률 대신 최종 클래스를 제출하세요.
- 행 수: 10,000.
- id는 R20000~R29999의 문자열입니다.

## 100점 환산
`score = 100 × Accuracy` (모든 테스트 행에서 정확히 일치한 클래스의 비율). 세 문제 합계는 300점입니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
DATA_DIR = Path("data")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print("train:", train.shape, "test:", test.shape)
train.head()

## TODO
데이터를 확인하고 전처리·모델·검증 방법을 구현하세요. 아래는 원문에서 제공한 간단한 베이스라인입니다. 필요에 맞게 수정하세요.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from scipy.sparse import hstack
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.base import BaseEstimator, ClassifierMixin

X, y, X_test = train.review.values, train.label.values, test.review.values
assert train.review.notna().all() and test.review.notna().all()

class NBLR(BaseEstimator, ClassifierMixin):
    """NB-SVM 계열: 이진 n-gram(글자 1~4, 어절 1~2)에 NB 로그비율 r 을 곱한 특징 + 로지스틱 회귀"""
    def __init__(self, C=8, alpha=1.0):
        self.C = C; self.alpha = alpha
    def _vec(self):
        return [CountVectorizer(analyzer="char_wb", ngram_range=(1, 4), binary=True, min_df=2),
                CountVectorizer(analyzer="word", ngram_range=(1, 2), binary=True, token_pattern=r"\S+", min_df=2)]
    def fit(self, T, y):
        self.v_ = [v.fit(T) for v in self._vec()]
        M = hstack([v.transform(T) for v in self.v_]).tocsr()
        p = self.alpha + M[y == 1].sum(0); q = self.alpha + M[y == 0].sum(0)
        self.r_ = np.asarray(np.log((p / p.sum()) / (q / q.sum()))).ravel()
        self.lr_ = LogisticRegression(C=self.C, max_iter=3000).fit(M.multiply(self.r_).tocsr(), y)
        self.classes_ = self.lr_.classes_
        return self
    def predict_proba(self, T):
        M = hstack([v.transform(T) for v in self.v_]).tocsr()
        return self.lr_.predict_proba(M.multiply(self.r_).tocsr())

def build():
    return {
        "cnb":  make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), sublinear_tf=True, min_df=2), ComplementNB(alpha=0.5)),
        "mnb":  make_pipeline(CountVectorizer(analyzer="char_wb", ngram_range=(1, 4), binary=True, min_df=2), MultinomialNB(alpha=0.5)),
        "nblr": NBLR(C=8),
    }

rank = lambda p: pd.Series(p).rank().values / len(p)        # 모델마다 확률 보정이 달라 순위 평균으로 결합
def to_label(score, pos_rate):                               # 상위 pos_rate 비율을 긍정(1)로 (문제: 긍정/부정 거의 1:1)
    return (score >= np.quantile(score, 1 - pos_rate)).astype(int)

# ---- 5-Fold CV ----
oof = {k: np.zeros(len(X)) for k in build()}
for a, b in StratifiedKFold(5, shuffle=True, random_state=42).split(X, y):
    for k, m in build().items():
        oof[k][b] = m.fit(X[a], y[a]).predict_proba(X[b])[:, 1]
for k in oof: print(f"{k:5s} CV acc: {((oof[k] >= .5) == y).mean():.4f}")
ens = np.mean([rank(p) for p in oof.values()], axis=0)
cv_acc = float((to_label(ens, y.mean()) == y).mean()); print(f"ensemble(rank avg) CV acc: {cv_acc:.4f} -> 환산 점수 약 {100 * cv_acc:.1f}")

# ---- 전체 train 재학습 후 test 예측 ----
score = np.mean([rank(m.fit(X, y).predict_proba(X_test)[:, 1]) for m in build().values()], axis=0)
pred = to_label(score, y.mean())
print("test 예측 긍정 비율: %.3f (train %.3f)" % (pred.mean(), y.mean()))

In [ ]:
submission = pd.DataFrame({"id": test["id"], "label": pred.astype(int)})
submission.to_csv("submission.csv", index=False)

In [ ]:
submission = pd.read_csv("submission.csv")
print(submission.shape)
assert len(submission) == len(test)
assert submission["id"].is_unique
assert set(submission["id"]) == set(test["id"])
submission.head()
assert list(submission.columns) == ["id", "label"] and set(submission.label.unique()) <= {0, 1} and submission.label.dtype.kind == "i"